In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from collections import Counter
import json
from scipy.stats import bootstrap

import decision_infovalue

# Data Preparation

In [2]:
data, metadata = decision_infovalue.get_dataset("cxr")
cxr_record_list = pd.read_csv("cxr-record-list.csv")
data_w_path = pd.merge(data, cxr_record_list[["study_id", "path"]], on="study_id", how="left")
data_w_path.loc[:, "path"] = data_w_path["path"].apply(lambda x: x.split(".")[0] + ".tfrecord")

# Filter dataset to only include rows where the tfrecord file exists
data_w_path = data_w_path[data_w_path["path"].apply(lambda x: os.path.exists(os.path.join("./generalized-image-embeddings-for-the-mimic-chest-x-ray-dataset-1.0", x)))]
data_w_path = data_w_path.reset_index(drop=True)
data_w_path = data_w_path.groupby("study_id").first().reset_index()
data_w_path

,study_id,Unnamed: 0,abnormal_stratified,diagnosis,densenet121_pred,inception_v3_pred,swin_b_pred,resnet50_pred,vit_b_16_pred,path
0,50001372,7042,False,0.0,0.1,0.1,0.1,0.2,0.0,files/p14/p14740030/s50001372/50d7d528-db58e77...
1,50001867,7938,True,0.0,0.1,0.2,0.1,0.2,0.1,files/p15/p15649581/s50001867/27777401-52524d2...
2,50001886,3554,False,1.0,0.3,0.4,0.2,0.1,0.5,files/p11/p11327070/s50001886/55d370ca-bdd3653...
3,50005580,4709,False,1.0,0.9,0.6,0.7,0.8,0.8,files/p12/p12456080/s50005580/eb297ae9-801aa64...
4,50005852,3601,False,0.0,0.4,0.4,0.6,0.5,0.4,files/p11/p11380379/s50005852/0b09373e-3281b2e...
...,...,...,...,...,...,...,...,...,...,...
12140,59996255,6416,False,0.0,0.1,0.2,0.1,0.1,0.1,files/p14/p14153350/s59996255/ecde50ec-410b094...
12141,59996457,11337,False,0.0,0.1,0.1,0.1,0.1,0.1,files/p19/p19131119/s59996457/8d6ff7f3-a2216a0...
12142,59996905,4571,False,0.0,0.1,0.1,0.0,0.0,0.0,files/p12/p12309846/s59996905/f6265921-4d28c20...
12143,59998636,7806,True,0.0,0.8,0.5,0.7,0.5,0.7,files/p15/p15526064/s59998636/9eba9527-51c1a4d...


We use the embedding from the CXR foundation model for MIMIC-CXR X ray images. Please download the data from [PhysioNet](https://physionet.org/content/image-embeddings-mimic-cxr/1.0/) and put in the current directory.

In [3]:
def read_tfrecord_manually(tfrecord_path):
    """Reads a TFRecord file and parses each record manually."""
    # Create an iterator that yields raw serialized records
    for raw_record in tf.compat.v1.io.tf_record_iterator(tfrecord_path):
        # Create an empty Example proto object
        example = tf.train.Example()

        # Parse the raw bytes into the Example object
        example.ParseFromString(raw_record)

    return example

import tensorflow as tf
import os

tfrecord_path = [os.path.join("./generalized-image-embeddings-for-the-mimic-chest-x-ray-dataset-1.0", data_w_path.loc[i, "path"]) for i in range(len(data_w_path))]

X = np.array([read_tfrecord_manually(p).features.feature['embedding'].float_list.value for p in tfrecord_path])
y = data_w_path["abnormal_stratified"].values


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


CancelledError: generalized-image-embeddings-for-the-mimic-chest-x-ray-dataset-1.0/files/p18/p18414625/s50006862/8e43dae5-7a447acc-cc8c9ea3-da80afb4-d150f533.tfrecord; Operation canceled

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score

# test_indices = data_w_path.sample(frac=0.3, random_state=42).index

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


## Neural Network

In [ ]:
# Create and train the neural network
mlp = MLPClassifier(hidden_layer_sizes=(256, 64), max_iter=500, random_state=42, verbose=False, early_stopping=True)
mlp.fit(X_train, y_train)

# Make predictions
y_pred = mlp.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
y_pred_proba = mlp.predict_proba(X_test)[:, 1]
brier_score = 1 - np.mean((y_pred_proba - y_test) ** 2)
print(f"Brier Score (Neural Network): {brier_score:.4f}")
print(f"Accuracy (Neural Network): {accuracy:.4f}")
print("\nClassification Report (Neural Network):")
print(classification_report(y_test, y_pred))


In [ ]:
# Calculate the calibration error (Expected Calibration Error, ECE)
from sklearn.calibration import calibration_curve

# We'll use 10 bins for the calibration curve
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=10, strategy="quantile")

# Expected Calibration Error (ECE)
ece = np.abs(prob_true - prob_pred).mean()
print(f"Expected Calibration Error (ECE) (Neural Network): {ece:.4f}")

In [ ]:
full_info_nn = 1 - np.mean((mlp.predict_proba(X)[:, 1] - y) ** 2)
no_info_nn = 1 - np.mean((np.mean(y) - y) ** 2)
print(f"The full info value using the neural network as rational belief estimator is {full_info_nn - no_info_nn:.4f}")

## Gradient Boost Method

In [ ]:
from xgboost import XGBClassifier

# Create and train the XGBoost classifier
xgb_clf = XGBClassifier(eval_metric='logloss', random_state=42, early_stopping_rounds=10)
X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(X_train, y_train, test_size=0.1, random_state=42)
xgb_clf.fit(X_train_xgb, y_train_xgb, eval_set=[(X_test_xgb, y_test_xgb)], verbose=False)

# Make predictions
y_pred_xgb = xgb_clf.predict(X_test)

# Evaluate the model
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
y_pred_proba_xgb = xgb_clf.predict_proba(X_test)[:, 1]
brier_score_xgb = 1 - np.mean((y_pred_proba_xgb - y_test) ** 2)
print(f"Brier Score (XGBoost): {brier_score_xgb:.4f}")
print(f"Accuracy (XGBoost): {accuracy_xgb:.4f}")
print("\nClassification Report (XGBoost):")
print(classification_report(y_test, y_pred_xgb))

# We'll use 10 bins for the calibration curve
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba_xgb, n_bins=10, strategy="quantile")

# Expected Calibration Error (ECE)
ece = np.abs(prob_true - prob_pred).mean()
print(f"Expected Calibration Error (ECE) (XGBoost): {ece:.4f}")


In [ ]:
full_info_xgb = 1 - np.mean((xgb_clf.predict_proba(X)[:, 1] - y) ** 2)
no_info_xgb = 1 - np.mean((np.mean(y) - y) ** 2)
print(f"The full info value using the XGBoost as rational belief estimator is {full_info_xgb - no_info_xgb:.4f}")

## Linear regression

In [ ]:
from sklearn.linear_model import LinearRegression

# Fit a linear regression model
linreg = LinearRegression()
linreg.fit(X_train, y_train)

# Make predictions
y_pred_linreg = linreg.predict(X_test)

# Evaluate the model
from sklearn.metrics import mean_squared_error, r2_score

mse_linreg = mean_squared_error(y_test, y_pred_linreg)
r2_linreg = r2_score(y_test, y_pred_linreg)
brier_score_linreg = 1 - np.mean((y_pred_linreg - y_test) ** 2)
accuracy_linreg = accuracy_score(y_test, y_pred_linreg > 0.5)

print(f"Mean Squared Error (Linear Regression): {mse_linreg:.4f}")
print(f"R^2 Score (Linear Regression): {r2_linreg:.4f}")
print(f"Brier Score (Linear Regression): {brier_score_linreg:.4f}")
print(f"Accuracy (Linear Regression): {accuracy_linreg:.4f}")
print("\nClassification Report (Linear Regression):")
print(classification_report(y_test, y_pred_linreg > 0.5))

y_pred_linreg = np.clip(y_pred_linreg, 0, 1)

# We'll use 10 bins for the calibration curve
prob_true, prob_pred = calibration_curve(y_test, y_pred_linreg, n_bins=10, strategy="quantile")

# Expected Calibration Error (ECE)
ece = np.abs(prob_true - prob_pred).mean()
print(f"Expected Calibration Error (ECE) (Linear Regression): {ece:.4f}")

In [ ]:
full_info_linreg = 1 - np.mean((linreg.predict(X) - y) ** 2)
no_info_linreg = 1 - np.mean((np.mean(y) - y) ** 2)
print(f"The full info value using the linear regression as rational belief estimator is {full_info_linreg - no_info_linreg:.4f}")